# LingBot-VLA：Colab A100 80 GB，10 个任务全量微调

这份 Notebook 有 **5 个主要代码单元格**，按顺序运行即可。不需要运行旧 Modal Notebook，
不需要旧 LoRA。每个任务使用官方 Aloha-AgileX clean50；10 个任务共 500 条示范。

**运行前：**在「运行时 → 更改运行时类型」选择 A100 GPU，能选择时启用高 RAM。
本配置要求实际可用显存约 80 GB，不适用于 A100 40 GB。Google Drive 需有**约 120 GB 可用存储**，
用来容纳最新完整续训状态、最佳 BF16 模型以及保存期间的新旧副本；Colab Pro 本身不等于获得这些 Drive 容量。
本地磁盘也需容纳约 24 GiB 基础权重、RGB 缓存和数据 ZIP，以本格输出的实际空间为准。

训练范围是视觉编码器、VLM、动作专家和动作投影等参与策略计算的主干；
不使用旧 adapter，不加载深度/视频教师，不宣称复现官方完整教师训练配方。
已做本地逻辑检查；**尚未在你的 Colab A100 实际运行此全量版本**。


## 1．连接存储、设置任务和训练参数

这里是日常需要修改的主要位置。默认 10 个任务均衡抽样；每任务 45 条训练、5 条离线验证，
不是把同一条轨迹拆成训练帧和验证帧。10,000 指优化器更新次数，实际会处理 160,000 个样本窗口。
没有人为训练时间上限，但 Colab 自身仍有资源/会话限制。

第一次保留 `RESUME = "auto"`，没有 checkpoint 时从官方模型开始；中断重连后，
**保持同一 RUN_NAME，重新运行 1→4 即可接续**。想重新开一轮实验时修改 RUN_NAME。


In [ ]:
# 改参数通常只需要改这一格。
from pathlib import Path
import json, re, shutil, subprocess
from google.colab import drive
drive.mount('/content/drive')

TASKS = [
    'adjust_bottle',          # 瓶子姿态调整
    'pick_diverse_bottles',   # 多样瓶子抓取
    'pick_dual_bottles',      # 双臂瓶子操作
    'place_empty_cup',        # 杯子放置
    'place_can_basket',       # 罐子放入篮子
    'stack_blocks_two',       # 两个方块堆叠
    'stack_bowls_two',        # 两个碗堆叠
    'handover_block',         # 方块交接
    'open_laptop',            # 笔记本开合
    'press_stapler',          # 订书机按压
]
RUN_NAME = 'lingbot_full10_v1'
SAVE_ROOT = '/content/drive/MyDrive/lingbot_training'
LOCAL_ROOT = '/content/lingbot_full10'  # 本地 SSD：可重建，不当作永久存储。
RESUME = 'auto'

STEPS = 10000
MICRO_BATCH = 2             # 不足时自动降到 1；有效 batch 仍保持 16。
EFFECTIVE_BATCH = 16
SAVE_EVERY = 500
VALIDATE_EVERY = 500

assert len(TASKS) == len(set(TASKS)) == 10
assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_NAME)
assert MICRO_BATCH in [1, 2] and EFFECTIVE_BATCH % MICRO_BATCH == 0
assert STEPS > 0 and SAVE_EVERY > 0 and VALIDATE_EVERY > 0
assert Path(LOCAL_ROOT).resolve().is_relative_to(Path('/content'))
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True)
print('当前 GPU：', gpu.strip())
assert int(gpu.strip().splitlines()[0].rsplit(',', 1)[1].strip().split()[0]) >= 75000, '此版本面向 80 GB GPU'

CFG = dict(
    tasks=TASKS, run_name=RUN_NAME, save_root=SAVE_ROOT, local_root=LOCAL_ROOT, resume=RESUME,
    steps=STEPS, micro_batch=MICRO_BATCH, effective_batch=EFFECTIVE_BATCH,
    save_every=SAVE_EVERY, validate_every=VALIDATE_EVERY,
    validation_episodes=5, validation_windows_per_task=32,
    horizon=16, max_text_length=128, workers=2, seed=42, log_every=10,
    lr_vision=1e-6, lr_vlm=5e-6, lr_action=2e-5, weight_decay=.01, warmup_steps=300,
    base_revision='11c703bf6a5c1f45b3b69168482da11fdbba53d7',
    processor_revision='ebb281ec70b05090aa6165b016eac8ec08e71b17',
    source_revision='be969b8fd117fb70550c5d4bf4bc328211b5b1b6',
    data_revision='3dc3b798668feb99ac61cc9086d84cbcc3d79186',
)
WORK = Path(LOCAL_ROOT)
WORK.mkdir(parents=True, exist_ok=True)
SCRIPTS = WORK / 'scripts'
SCRIPTS.mkdir(exist_ok=True)
CONFIG = WORK / 'config.json'
CONFIG.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding='utf-8')
print('本地剩余空间 GiB：', round(shutil.disk_usage(WORK).free / 1024**3, 1))
print('永久结果目录：', Path(SAVE_ROOT) / RUN_NAME)
print(f'{len(TASKS)} 个任务；{STEPS} 次正式更新；有效 batch={EFFECTIVE_BATCH}。')


## 2．安装独立环境并写入训练程序

训练使用独立 Python 环境，避免与 Colab 原有包混装；不需要重启 Notebook 内核。
固定 Torch 2.8 / Transformers 4.57.3 等版本，并安装 8-bit 优化器。
**首次可能需要几分钟。**下面较长的辅助程序由本 Notebook 自带，通常不必编辑；
代码包里也提供单独的 `.py` 文件方便阅读。没有 20-step 试训，也不在这格加载大模型。


In [ ]:
#@title 安装独立环境并写入辅助程序（展开可查看完整源码）
# 本格自带全部辅助程序，不需要另传 .py，也不依赖之前的 Modal 文件。
import hashlib, os, sys, venv

# ---------- colab10_checkpoint.py ----------
(SCRIPTS / 'colab10_checkpoint.py').write_text(r'''"""分片保存完整模型/优化器。只有写完 COMPLETE.json 才切换续训指针。"""
import os
from pathlib import Path

from colab10_common import digest, read_json, remove_owned, write_json


def tree_to(value, device):
    import torch
    if isinstance(value, torch.Tensor):
        return value.detach().to(device=device, copy=True)
    if isinstance(value, dict):
        return {k: tree_to(v, device) for k, v in value.items()}
    if isinstance(value, list):
        return [tree_to(v, device) for v in value]
    if isinstance(value, tuple):
        return tuple(tree_to(v, device) for v in value)
    return value


def tree_bytes(value):
    import torch
    if isinstance(value, torch.Tensor):
        return value.numel() * value.element_size()
    if isinstance(value, dict):
        return sum(tree_bytes(v) for v in value.values())
    if isinstance(value, (list, tuple)):
        return sum(tree_bytes(v) for v in value)
    return 0


def save_weights(model, folder, dtype=None, shard_bytes=1024**3):
    from safetensors.torch import save_file
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    part, size, index, total, number = {}, 0, {}, 0, 0

    def flush():
        nonlocal part, size, number
        if not part:
            return
        number += 1
        name = f'model-{number:05d}.safetensors'
        save_file(part, str(folder / name), metadata={'format': 'pt'})
        index.update({k: name for k in part})
        part, size = {}, 0

    for name, tensor in model.state_dict().items():
        target_dtype = dtype if dtype is not None and tensor.is_floating_point() else tensor.dtype
        bytes_ = tensor.numel() * __import__('torch').empty((), dtype=target_dtype).element_size()
        if part and size + bytes_ > shard_bytes:
            flush()
        part[name] = tensor.detach().to(device='cpu', dtype=target_dtype, copy=True).contiguous()
        size += bytes_
        total += bytes_
    flush()
    write_json(folder / 'model.safetensors.index.json',
               {'metadata': {'total_size': total}, 'weight_map': index})


def save_optimizer(optimizer, folder, shard_bytes=1024**3):
    import torch
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    state_dict = optimizer.state_dict()  # 引用 GPU 状态，不在这里复制整套。
    part, size, files = {}, 0, []

    def flush():
        nonlocal part, size
        if not part:
            return
        name = f'optimizer-{len(files) + 1:05d}.pt'
        torch.save(part, folder / name)
        files.append(name)
        part, size = {}, 0

    for key, state in state_dict['state'].items():
        bytes_ = tree_bytes(state)
        if part and size + bytes_ > shard_bytes:
            flush()
        part[key] = tree_to(state, 'cpu')
        size += bytes_
    flush()
    write_json(folder / 'optimizer_index.json',
               {'files': files, 'param_groups': state_dict['param_groups']})


def load_optimizer(optimizer, folder):
    import torch
    folder = Path(folder)
    index = read_json(folder / 'optimizer_index.json')
    assert len(index['param_groups']) == len(optimizer.param_groups)
    mapping = {}
    for current, saved in zip(optimizer.param_groups, index['param_groups']):
        assert current['names'] == saved['names'], '优化器参数顺序变化，不能续训'
        mapping.update(zip(saved['params'], current['params']))
        current.update({k: v for k, v in saved.items() if k != 'params'})
    optimizer.state.clear()
    for filename in index['files']:
        chunk = torch.load(folder / filename, map_location='cpu', weights_only=True)
        for key, state in chunk.items():
            parameter = mapping[key]
            optimizer.state[parameter] = tree_to(state, parameter.device)
        del chunk
    # 保持 bitsandbytes 原始 dtype（尤其 uint8 状态）；下一次 step 正常初始化管理器。


def finalize(folder, metadata):
    folder = Path(folder)
    files = {p.relative_to(folder).as_posix(): {'bytes': p.stat().st_size, 'sha256': digest(p)}
             for p in folder.rglob('*') if p.is_file() and p.name != 'COMPLETE.json'}
    write_json(folder / 'COMPLETE.json', {**metadata, 'files': files})


def verify(folder):
    folder = Path(folder)
    complete = read_json(folder / 'COMPLETE.json')
    for name, record in complete['files'].items():
        path = (folder / name).resolve()
        assert path.is_relative_to(folder.resolve())
        assert path.stat().st_size == record['bytes'] and digest(path) == record['sha256'], f'损坏的 checkpoint: {name}'
    return complete


def resolve_pointer(run, kind):
    run = Path(run).resolve()
    pointer = run / f'{kind}.json'
    if not pointer.is_file():
        return None
    folder = (run / read_json(pointer)['directory']).resolve()
    assert folder.is_relative_to(run) and folder != run
    return folder


def publish(run, folder, kind, step):
    run, folder = Path(run), Path(folder)
    assert (folder / 'COMPLETE.json').is_file()
    previous = resolve_pointer(run, kind)
    final = folder.with_name(folder.name.removesuffix('.incomplete'))
    assert not final.exists()
    os.replace(folder, final)
    write_json(run / f'{kind}.json', {'directory': final.name, 'step': step})
    if previous is not None and previous != final:
        remove_owned(previous, run)
    return final
''', encoding='utf-8')

# ---------- colab10_common.py ----------
(SCRIPTS / 'colab10_common.py').write_text(r'''"""Colab 十任务训练的公共函数。只在选定项目目录内写入。"""
import hashlib
import json
import os
import random
import shutil
from pathlib import Path

CAMERAS = ['head_camera', 'left_camera', 'right_camera']
SLOTS = [0, 1, 2, 3, 4, 5, 28, 6, 7, 8, 9, 10, 11, 29]
FORMAT = 'robotwin_rgb224_next_joint_v1'


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(temporary, path)


def digest(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b''):
            result.update(block)
    return result.hexdigest()


def identity(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def remove_owned(path, parent):
    """仅清理当前项目自己的临时文件/旧 checkpoint，不接收任意目录。"""
    path, parent = Path(path).resolve(), Path(parent).resolve()
    if path == parent or not path.is_relative_to(parent):
        raise ValueError(f'拒绝清理项目目录以外的路径：{path}')
    if path.is_dir():
        shutil.rmtree(path)
    elif path.exists():
        path.unlink()


def pack(values, width=55):
    import numpy as np
    result = np.zeros((*values.shape[:-1], width), dtype=np.float32)
    result[..., SLOTS] = values
    return result


def split_episodes(task, count, seed):
    import numpy as np
    # 与任务列表顺序无关：换排列不改变每个任务的验证轨迹。
    salt = int(hashlib.sha256(task.encode()).hexdigest()[:8], 16)
    order = np.random.default_rng([seed, salt]).permutation(50)
    valid = sorted(order[:count].tolist())
    return sorted(order[count:].tolist()), valid


def selections(manifest, cfg, step):
    """每个 update 确定性抽样：任务→轨迹→起始帧，避免长轨迹独占。"""
    import numpy as np
    rng = np.random.default_rng([cfg['seed'], step])
    tasks = cfg['tasks']
    # 连续的抽样槽位轮流覆盖任务；10 个任务时连续 5 次更新各取 8 个样本。
    ids = (np.arange(cfg['effective_batch']) + (step - 1) * cfg['effective_batch']) % len(tasks)
    rng.shuffle(ids)
    result = []
    for position, task_id in enumerate(ids):
        task = tasks[int(task_id)]
        episodes = manifest['tasks'][task]['train_episodes']
        episode = int(rng.choice(episodes))
        n = manifest['tasks'][task]['episodes'][str(episode)]['frames']
        result.append((task, episode, int(rng.integers(n)),
                       int(rng.integers(0, 2**32 - 1))))
    return result


def augment_rgb(views, seed):
    """只增广未归一化的 RGB；三相机共享颜色参数，不改变几何和动作标签。"""
    import numpy as np
    from PIL import Image, ImageEnhance, ImageFilter
    rng = np.random.default_rng(seed)
    if rng.random() < 0.5:
        return np.array(views, copy=True)
    brightness, contrast = rng.uniform(.85, 1.15, 2)
    saturation = rng.uniform(.9, 1.1)
    blur = rng.random() < .1
    sigma = rng.uniform(1., 3.) if rng.random() < .15 else 0.
    output = []
    for view in views:
        image = Image.fromarray(view)
        image = ImageEnhance.Brightness(image).enhance(float(brightness))
        image = ImageEnhance.Contrast(image).enhance(float(contrast))
        image = ImageEnhance.Color(image).enhance(float(saturation))
        if blur:
            image = image.filter(ImageFilter.GaussianBlur(radius=.35))
        rgb = np.asarray(image).astype(np.float32)
        if sigma:
            rgb += rng.normal(0, sigma, rgb.shape)
        output.append(rgb.round().clip(0, 255).astype(np.uint8))
    return np.stack(output)


class EpisodeStore:
    """图像 mmap 按需读本地 SSD；不把 500 条轨迹预装进 RAM/GPU。"""
    def __init__(self, root, manifest, stats, horizon):
        self.root, self.manifest, self.stats, self.horizon = Path(root), manifest, stats, horizon
        self.arrays = {}

    def sample(self, key, augmentation=True):
        import numpy as np
        task, episode, t, seed = key
        name = (task, episode)
        if name not in self.arrays:
            folder = self.root / 'prepared' / task / f'episode{episode}'
            numeric = np.load(folder / 'joints.npz', allow_pickle=False)
            self.arrays[name] = (np.load(folder / 'rgb.npy', mmap_mode='r'),
                                 numeric['states'], numeric['actions'])
        rgb, states, actions = self.arrays[name]
        offsets = t + np.arange(self.horizon)
        future = np.minimum(offsets, len(actions) - 1)
        s = self.stats
        state = (states[t] - s['s_low']) / (np.array(s['s_high']) - s['s_low'] + 1e-6) * 2 - 1
        action = (actions[future] - s['a_low']) / (np.array(s['a_high']) - s['a_low'] + 1e-6) * 2 - 1
        mask = pack(np.ones((self.horizon, 14), dtype=np.float32))
        mask[offsets >= len(actions)] = 0
        prompts = self.manifest['tasks'][task]['episodes'][str(episode)]['seen']
        text = prompts[int(seed) % len(prompts)]
        return {'rgb': augment_rgb(rgb[t], seed) if augmentation else np.array(rgb[t], copy=True),
                'state': pack(state), 'actions': pack(action), 'joint_mask': mask,
                'instruction': text, 'task': task, 'seed': seed}


class WindowDataset:
    def __init__(self, root, manifest, stats, horizon):
        self.store = EpisodeStore(root, manifest, stats, horizon)

    def __getitem__(self, key):
        return self.store.sample(key)


class UpdateSampler:
    def __init__(self, manifest, cfg, first_step, micro):
        self.manifest, self.cfg, self.first_step, self.micro = manifest, cfg, first_step, micro

    def __iter__(self):
        for step in range(self.first_step, self.cfg['steps'] + 1):
            chosen = selections(self.manifest, self.cfg, step)
            for start in range(0, len(chosen), self.micro):
                yield chosen[start:start + self.micro]

    def __len__(self):
        return max(0, self.cfg['steps'] - self.first_step + 1) * self.cfg['effective_batch'] // self.micro


class BatchCollator:
    def __init__(self, processor_dir, max_length):
        self.processor_dir, self.max_length = str(processor_dir), max_length
        self.processor = None

    def __call__(self, samples):
        import numpy as np
        import torch
        if self.processor is None:
            from transformers import AutoProcessor
            self.processor = AutoProcessor.from_pretrained(self.processor_dir, local_files_only=True)
        p = self.processor
        images = p.image_processor(images=[v for s in samples for v in s['rgb']],
                                   do_resize=False, return_tensors='pt')
        prompts = [p.tokenizer.apply_chat_template(
            [{'role': 'user', 'content': s['instruction']}], tokenize=False,
            add_generation_prompt=False) for s in samples]
        # 不静默截断任务指令：超长时明确提示增加 max_text_length。
        encoded = p.tokenizer(prompts, add_special_tokens=False, padding='max_length',
                              truncation=False, max_length=self.max_length, return_tensors='pt')
        if encoded['input_ids'].shape[1] > self.max_length:
            raise ValueError('官方指令超出 max_text_length，请在配置中增加长度')
        b = len(samples)
        pixels, grid = images['pixel_values'], images['image_grid_thw']
        assert torch.all(grid == grid[0])
        return {'images': pixels.reshape(b, 3, -1, pixels.shape[-1]),
                'image_grid_thw': grid.reshape(b, 3, 3),
                'img_masks': torch.ones(b, 3, dtype=torch.bool),
                'lang_tokens': encoded['input_ids'], 'lang_masks': encoded['attention_mask'].bool(),
                **{k: torch.from_numpy(np.stack([s[k] for s in samples]))
                   for k in ['state', 'actions', 'joint_mask']}}


def seed_all(seed):
    import numpy as np
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
''', encoding='utf-8')

# ---------- colab10_model.py ----------
(SCRIPTS / 'colab10_model.py').write_text(r'''"""固定版本 LingBot 策略主干：SDPA、可反传 MoE、联合层梯度检查点。

不注入 LoRA，不加载旧 adapter，不加载深度/视频教师。
FP32 可训练权重 + BF16 autocast；8-bit 优化器在训练入口创建。
"""
import functools
import ast
import shutil
import struct
import sys
import types
from pathlib import Path

from colab10_common import read_json


def defer_official_data_imports(path):
    """只需 data.constants 时不提前导入 LeRobot；保留原始公开导出的按需访问。"""
    original = Path(path).read_text(encoding='utf-8')
    tree = ast.parse(original)
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == '_COLAB_DEFERRED_EXPORTS' for t in node.targets):
            assert 'build_vla_dataset' in ast.literal_eval(node.value)
            return
    exports = {}
    for node in tree.body:
        if isinstance(node, ast.ImportFrom) and node.level:
            for alias in node.names:
                assert alias.name != '*'
                exports[alias.asname or alias.name] = ('.' * node.level + node.module, alias.name)
    assert 'build_vla_dataset' in exports and 'build_dataloader' in exports
    header = original.split('from .chat_template import build_chat_template')[0]
    lazy = ("from importlib import import_module\n"
            f"_COLAB_DEFERRED_EXPORTS = {exports!r}\n"
            "__all__ = list(_COLAB_DEFERRED_EXPORTS)\n\n"
            "def __getattr__(name):\n"
            "    if name not in _COLAB_DEFERRED_EXPORTS:\n"
            "        raise AttributeError(name)\n"
            "    module, attribute = _COLAB_DEFERRED_EXPORTS[name]\n"
            "    value = getattr(import_module(module, __name__), attribute)\n"
            "    globals()[name] = value\n"
            "    return value\n")
    Path(path).write_text(header + '# Colab NPY loader: defer unused dataset dependencies.\n' + lazy, encoding='utf-8')


def sdpa_joint_attention(query, key, value, attention_mask, scaling=None):
    import torch
    import torch.nn.functional as F
    dtype = torch.bfloat16 if query.is_cuda else query.dtype
    q, k, v = [x.transpose(1, 2).to(dtype) for x in (query, key, value)]
    assert q.shape[1] % k.shape[1] == 0
    if q.shape[1] != k.shape[1]:
        repeats = q.shape[1] // k.shape[1]
        k, v = k.repeat_interleave(repeats, 1), v.repeat_interleave(repeats, 1)
    mask = attention_mask[:, None] if attention_mask is not None and attention_mask.ndim == 3 else attention_mask
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0., scale=scaling)
    return out.transpose(1, 2).contiguous().reshape(query.shape[0], query.shape[1], -1).to(query.dtype)


def torch_moe(self, module, num_experts, routing_weights, selected_experts, hidden_states):
    import torch
    import torch.nn.functional as F
    output = torch.zeros_like(hidden_states)
    # unbind 让每个大参数只汇总一次各专家的梯度，保持官方 fused 权重布局。
    weights = zip(self.gate_proj.unbind(0), self.up_proj.unbind(0), self.down_proj.unbind(0))
    for expert, (gate_weight, up_weight, down_weight) in enumerate(weights):
        rows, slots = torch.where(selected_experts == expert)
        if rows.numel() == 0:
            continue
        x = hidden_states[rows]
        y = F.linear(F.silu(F.linear(x, gate_weight)) * F.linear(x, up_weight), down_weight)
        output = output.index_add(0, rows, y.to(output.dtype) * routing_weights[rows, slots, None])
    return output


def checkpointed_joint_forward(self, attention_mask=None, position_ids=None, vlm_position_ids=None,
                               past_key_values=None, inputs_embeds=None, use_cache=None,
                               fill_kv_cache=None, ada_cond=None, visual_pos_masks=None,
                               deepstack_visual_embeds=None):
    """对官方联合 VLM/action 层整体重计算，包含层间联合注意力。"""
    import torch
    from torch.utils.checkpoint import checkpoint
    # 验证和后续推理沿用官方路径，包括 prefix-only / suffix-only KV cache。
    if not self.training or not torch.is_grad_enabled():
        return self._colab_original_forward(
            attention_mask=attention_mask, position_ids=position_ids, vlm_position_ids=vlm_position_ids,
            past_key_values=past_key_values, inputs_embeds=inputs_embeds, use_cache=use_cache,
            fill_kv_cache=fill_kv_cache, ada_cond=ada_cond, visual_pos_masks=visual_pos_masks,
            deepstack_visual_embeds=deepstack_visual_embeds)
    assert not use_cache and past_key_values is None
    assert all(x is not None for x in inputs_embeds)
    models = [self.qwenvl.model.language_model, self.qwen_expert.model]
    hidden_vlm, hidden_action = inputs_embeds
    router_logits_list = []

    def layer(vlm, action, layer_idx):
        hidden = [vlm, action]
        qs, ks, vs = [], [], []
        for i, x in enumerate(hidden):
            kwargs = {'ada_cond': ada_cond} if i == 1 else {}
            q, k, v = models[i].layers[layer_idx](x, compute_kqv=True, **kwargs)
            qs.append(q.float())
            ks.append(k.float())
            vs.append(v.float())
        q, k = self.apply_mrope(torch.cat(qs, 1), torch.cat(ks, 1), position_ids)
        attended = self.attention_interface(q, k, torch.cat(vs, 1), attention_mask)
        boundary = vlm.shape[1]
        out_vlm = models[0].layers[layer_idx](vlm, attended, 0, boundary, output_atten=True)
        out_vlm = self._apply_deepstack(out_vlm, layer_idx, visual_pos_masks, deepstack_visual_embeds)
        out_action, logits = models[1].layers[layer_idx](
            action, attended, boundary, boundary + action.shape[1], output_atten=True, ada_cond=ada_cond)
        return out_vlm, out_action, logits

    assert len(models[0].layers) == len(models[1].layers)
    for index in range(len(models[0].layers)):
        # partial 固定层号，避免反向重计算时所有闭包都变成最后一层。
        hidden_vlm, hidden_action, logits = checkpoint(
            functools.partial(layer, layer_idx=index), hidden_vlm, hidden_action,
            use_reentrant=False, preserve_rng_state=True)
        if logits is not None:
            router_logits_list.append(logits)
    out_vlm = models[0].norm(hidden_vlm)
    if self.config.final_norm_adanorm:
        out_action, _ = models[1].norm(hidden_action, ada_cond)
    else:
        out_action = models[1].norm(hidden_action)
    return [out_vlm, out_action], None, router_logits_list


def shape_index(folder):
    import json
    index = read_json(folder / 'model.safetensors.index.json')
    shapes = {}
    for shard in sorted(set(index['weight_map'].values())):
        with (folder / shard).open('rb') as file:
            length = struct.unpack('<Q', file.read(8))[0]
            assert 0 < length < 64 * 1024**2
            header = json.loads(file.read(length))
        shapes.update({k: v['shape'] for k, v in header.items() if k != '__metadata__'})
    return index, shapes


def build_model(cfg, root, weights=None, saved_config=None):
    import torch
    import yaml
    from accelerate import init_empty_weights
    from safetensors import safe_open
    root = Path(root)
    canonical = root / 'source/official'
    source = root / 'source/runtime'
    # 每次从固定的官方源码生成运行副本，绝不在已补丁的文本上重复替换。
    shutil.copytree(canonical, source, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('__pycache__', '*.pyc', '.git'))
    defer_official_data_imports(source / 'lingbotvla/data/__init__.py')
    path = source / 'lingbotvla/models/vla/lingbot_vla/modeling_lingbot_vla_v2.py'
    text = path.read_text(encoding='utf-8')
    for key in ['vlm_config', 'vlm_config.text_config', 'self.config.qwen_expert_config']:
        old = f'{key}._attn_implementation = "flash_attention_2"'
        assert text.count(old) == 1, f'官方源码版本不匹配：{old}'
        text = text.replace(old, f'{key}._attn_implementation = "eager"')
    path.write_text(text, encoding='utf-8')
    sys.path.insert(0, str(source))
    from lingbotvla.models.vla.lingbot_vla.qwen3vl_in_vla import apply_lingbot_qwen3_vl_patch
    from lingbotvla.models.vla.lingbot_vla.qwen2_action_expert import apply_lingbot_qwen2_patch, Qwen2FusedExperts
    from lingbotvla.models.vla.lingbot_vla.configuration_lingbot_vla import LingbotVLAV2Config
    from lingbotvla.models.vla.lingbot_vla.modeling_lingbot_vla_v2 import LingbotVlaV2Policy
    apply_lingbot_qwen3_vl_patch()
    apply_lingbot_qwen2_patch()
    Qwen2FusedExperts.forward = torch_moe
    folder = Path(weights) if weights else root / 'models/base'
    index, shapes = shape_index(folder)
    if saved_config:
        config_args = read_json(Path(saved_config) / 'config.json')
    else:
        template = yaml.safe_load((source / 'configs/vla/robotwin/robotwin.yaml').read_text())
        config_args = {**template['model'], **template['train']}
    config_args.update(
        tokenizer_path=str(root / 'models/processor'), vlm_repo_id=None,
        align_params={}, use_compile=False, use_cache=False, attention_implementation='eager',
        vit_attn_implementation='sdpa', moe_implementation='fused',
        freeze_vision_encoder=False, train_expert_only=False, action_fp32=False,
        chunk_size=cfg['horizon'], tokenizer_max_length=cfg['max_text_length'],
        # 保留官方 MoE 平衡和 z-loss；bias_update_speed=0 避免重计算修改路由。
        sequence_wise_loss_coeff=.001, router_z_loss_coeff=.0001, bias_update_speed=0,
        precompute_grid_thw=False, use_qwen3_fixed_grid_cache=False,
        action_dim=shapes['model.action_out_proj.weight'][0],
        max_action_dim=shapes['model.action_in_proj.weight'][1],
        max_state_dim=shapes['model.state_proj.weight'][1])
    config = LingbotVLAV2Config(**config_args)
    with init_empty_weights(include_buffers=False):
        model = LingbotVlaV2Policy(config)
    expected = model.state_dict()
    parameters = set(dict(model.named_parameters()))
    missing = parameters - set(shapes)
    mismatches = [k for k, v in expected.items() if k in shapes and list(v.shape) != shapes[k]]
    assert not missing and not mismatches, f'主干参数缺失/形状不匹配：{list(missing)[:5]}, {mismatches[:5]}'
    ignored = sorted(set(shapes) - set(expected))
    assert not any(k.startswith('model.qwenvl_with_expert.') for k in ignored), ignored[:5]
    assert all(any(x in k for x in ['depth', 'video', 'task', 'future', 'query', 'align']) for k in ignored), ignored[:5]
    loaded = set()
    for shard in sorted(set(index['weight_map'].values())):
        state = {}
        with safe_open(folder / shard, framework='pt', device='cpu') as file:
            for name in file.keys():
                if name in expected:
                    tensor = file.get_tensor(name)
                    # FP32 参数避免小学习率更新被 BF16 权重舍入吞掉。
                    state[name] = tensor.to(device='cuda', dtype=torch.float32) if tensor.is_floating_point() else tensor.cuda()
                    loaded.add(name)
        model.load_state_dict(state, strict=False, assign=True)
        del state
        print('已加载权重分片：', shard, flush=True)
    assert parameters <= loaded and not any(p.is_meta for p in model.parameters())
    del expected
    model.to('cuda')
    model.requires_grad_(True)
    joint = model.model.qwenvl_with_expert
    joint.attention_interface = sdpa_joint_attention
    joint._colab_original_forward = joint.forward
    joint.forward = types.MethodType(checkpointed_joint_forward, joint)
    # 只为 vision blocks 开启 HF checkpoint；联合文本/动作层由上面的整体包装处理。
    from torch.utils.checkpoint import checkpoint
    for block in joint.qwenvl.visual.blocks:
        block.gradient_checkpointing = True
        block._gradient_checkpointing_func = functools.partial(checkpoint, use_reentrant=False)
    # 关闭训练重计算中不参与损失的路由计数副作用；可微平衡损失保留。
    import lingbotvla.models.vla.lingbot_vla.qwen2_action_expert as expert_module
    expert_module._update_moe_runtime_stats = lambda *args, **kwargs: None
    # lm_head 不参与动作损失；不对它虚报全量更新。
    unused = []
    for name, parameter in model.named_parameters():
        if '.qwenvl.lm_head.' in name or '.qwen_expert.lm_head.' in name:
            parameter.requires_grad_(False)
            unused.append(name)
    model.train()
    metadata = {'ignored_inactive_auxiliary_keys': ignored, 'inactive_lm_heads': unused,
                'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
                'parameter_dtype': 'float32', 'compute_dtype': 'bfloat16',
                'checkpointing': 'joint VLM/action layers + vision blocks',
                'moe': 'differentiable torch experts with original fused weight layout',
                'teachers': False, 'lora': False}
    return model, metadata


def optimizer_groups(model, cfg):
    grouped = {}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if '.qwenvl.model.visual.' in name:
            role = 'vision'
        elif '.qwenvl.' in name:
            role = 'vlm'
        else:
            role = 'action'
        decay = parameter.ndim > 1 and not name.endswith('.bias') and 'norm' not in name.lower()
        group = grouped.setdefault((role, decay), {'params': [], 'names': [],
            'lr': cfg[f'lr_{role}'], 'weight_decay': cfg['weight_decay'] if decay else 0., 'role': role})
        group['params'].append(parameter)
        group['names'].append(name)
    return list(grouped.values())
''', encoding='utf-8')

# ---------- colab10_prepare.py ----------
(SCRIPTS / 'colab10_prepare.py').write_text(r'''"""下载固定版本的官方 clean 数据、转成本地训练缓存、计算训练集统计量。"""
import argparse
import os
import shutil
import tarfile
import urllib.request
import zipfile
from pathlib import Path

from colab10_common import (CAMERAS, FORMAT, digest, identity, read_json,
                            remove_owned, split_episodes, write_json)


def download_models(cfg, root):
    from huggingface_hub import HfApi, hf_hub_download, snapshot_download
    api = HfApi()
    records = {}
    for name, repo, revision in [
        ('base', 'robbyant/lingbot-vla-v2-6b', cfg['base_revision']),
        ('processor', 'Qwen/Qwen3-VL-4B-Instruct', cfg['processor_revision']),
    ]:
        info = api.model_info(repo, revision=revision, files_metadata=True)
        entries = {entry.rfilename: entry for entry in info.siblings}
        folder = root / 'models' / name
        support = [p for p in entries if '/' not in p and p.endswith(
            ('.json', '.txt', '.model', '.jinja', '.tiktoken'))
            and p != 'model.safetensors.index.json']
        if name == 'base':
            index_file = hf_hub_download(repo, 'model.safetensors.index.json',
                                         revision=revision, local_dir=folder)
            shards = set(read_json(index_file)['weight_map'].values())
            wanted = sorted(set(support) | shards | {'model.safetensors.index.json'})
        else:
            wanted = sorted(support)  # 配置和分词器，不下载第二套 Qwen 模型。
        remaining = sum(entries[p].size for p in wanted if not (folder / p).is_file()
                        or (folder / p).stat().st_size != entries[p].size)
        if shutil.disk_usage(root).free < remaining + 8 * 1024**3:
            raise RuntimeError('本地磁盘不足，请使用 Colab 高 RAM 运行时并释放本项目无用文件')
        print(f'下载 {name}: {repo}@{info.sha}，尚需 {remaining / 1024**3:.2f} GiB', flush=True)
        snapshot_download(repo, revision=revision, allow_patterns=wanted,
                          local_dir=folder, max_workers=2)
        for p in wanted:
            assert (folder / p).stat().st_size == entries[p].size, p
        records[name] = {'repo': repo, 'revision': info.sha, 'files': wanted}
    return records


def download_source(cfg, root):
    source = root / 'source' / 'official'
    marker = source / '_download.json'
    if marker.is_file() and read_json(marker)['revision'] == cfg['source_revision']:
        return read_json(marker)
    archive = root / 'source.tar.gz'
    url = f"https://codeload.github.com/Robbyant/lingbot-vla-v2/tar.gz/{cfg['source_revision']}"
    print('下载固定版本 LingBot 官方源码', flush=True)
    urllib.request.urlretrieve(url, archive)
    source.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive) as package:
        for member in package:
            relative = Path(*Path(member.name).parts[1:])
            target = (source / relative).resolve()
            if not target.is_relative_to(source.resolve()):
                raise ValueError('源码压缩包路径越界')
            if member.isfile():
                target.parent.mkdir(parents=True, exist_ok=True)
                with package.extractfile(member) as inp, target.open('wb') as out:
                    shutil.copyfileobj(inp, out)
    record = {'revision': cfg['source_revision'], 'archive_sha256': digest(archive)}
    write_json(marker, record)
    archive.unlink()
    return record


def read_states(file):
    import numpy as np
    left = np.asarray(file['joint_action/left_arm'], dtype=np.float32)
    right = np.asarray(file['joint_action/right_arm'], dtype=np.float32)
    n = len(left)
    assert n >= 2 and left.shape == right.shape == (n, 6)
    grips = [np.asarray(file[f'joint_action/{side}_gripper'], dtype=np.float32).reshape(n, 1)
             for side in ['left', 'right']]
    states = np.concatenate([left, grips[0], right, grips[1]], axis=1)
    assert states.shape == (n, 14) and np.isfinite(states).all()
    for camera in CAMERAS:
        assert len(file[f'observation/{camera}/rgb']) == n
    return states


def decode_rgb(value):
    import cv2
    import numpy as np
    from PIL import Image
    bits = value.tobytes() if hasattr(value, 'tobytes') else bytes(value)
    bgr = cv2.imdecode(np.frombuffer(bits, dtype=np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError('官方数据中的 RGB 解码失败')
    # 与已跑通的转换对齐：先统一为 640×480，再缩至模型的 224×224。
    bgr = cv2.resize(bgr, (640, 480), interpolation=cv2.INTER_LINEAR)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return np.asarray(Image.fromarray(rgb).resize((224, 224), Image.Resampling.BILINEAR))


def prepare_task(cfg, root, task):
    import h5py
    import numpy as np
    from huggingface_hub import HfApi, hf_hub_download
    repo = 'TianxingChen/RoboTwin2.0'
    filename = f'dataset/{task}/aloha-agilex_clean_50.zip'
    # 精确文件名白名单；不使用整个官方训练清单里的 randomized/Piper 数据。
    entries = list(HfApi().list_repo_tree(repo, path_in_repo=f'dataset/{task}',
                                         revision=cfg['data_revision'], repo_type='dataset'))
    selected = [entry for entry in entries if entry.path == filename]
    assert len(selected) == 1, f'没有指定的官方 Aloha clean50 文件：{filename}'
    entry = selected[0]
    if shutil.disk_usage(root).free < entry.size + 4 * 1024**3:
        raise RuntimeError(f'下载 {task} 前本地空间不足')
    archive = Path(hf_hub_download(repo, filename, repo_type='dataset',
                                 revision=cfg['data_revision'], local_dir=root / 'downloads'))
    assert archive.stat().st_size == entry.size
    archive_hash = digest(archive)
    lfs = getattr(entry, 'lfs', None)
    lfs_hash = getattr(lfs, 'sha256', None) if lfs is not None else None
    if isinstance(lfs, dict):
        lfs_hash = lfs.get('sha256')
    if lfs_hash:
        assert archive_hash == lfs_hash, 'ZIP 哈希不匹配'
    specification = {'format': FORMAT, 'archive_sha256': archive_hash,
                     'cameras': CAMERAS, 'size': [224, 224]}
    cache_id = identity(specification)
    destination = root / 'prepared' / task
    destination.mkdir(parents=True, exist_ok=True)
    episodes = {}
    with zipfile.ZipFile(archive) as package:
        names = set(package.namelist())
        for episode in range(50):
            prefix = 'aloha-agilex_clean_50'
            hp, jp = f'{prefix}/data/episode{episode}.hdf5', f'{prefix}/instructions/episode{episode}.json'
            assert hp in names and jp in names, f'{task}/episode{episode} 不完整'
            instructions = __import__('json').loads(package.read(jp))
            seen = instructions.get('seen')
            assert isinstance(seen, list) and seen and all(isinstance(s, str) and s.strip() for s in seen)
            folder = destination / f'episode{episode}'
            marker = folder / 'complete.json'
            if marker.is_file():
                record = read_json(marker)
                if record['cache_id'] != cache_id:
                    raise ValueError(f'旧缓存与当前数据不同：{folder}；请修改 local_root')
                assert (folder / 'rgb.npy').stat().st_size == record['rgb_bytes']
                assert digest(folder / 'joints.npz') == record['joints_sha256']
                episodes[str(episode)] = record
                continue
            temporary = destination / f'.episode{episode}.incomplete'
            if temporary.exists():
                remove_owned(temporary, destination)
            temporary.mkdir()
            hdf5 = temporary / 'source.hdf5'
            with package.open(hp) as inp, hdf5.open('wb') as out:
                shutil.copyfileobj(inp, out)
            with h5py.File(hdf5, 'r') as file:
                states = read_states(file)
                n = len(states) - 1
                needed = n * 3 * 224 * 224 * 3
                if shutil.disk_usage(root).free < needed + 8 * 1024**3:
                    raise RuntimeError('224 RGB 缓存将写满本地盘，请换更大的 Colab 运行时磁盘')
                rgb = np.lib.format.open_memmap(temporary / 'rgb.npy', mode='w+',
                                               dtype=np.uint8, shape=(n, 3, 224, 224, 3))
                for t in range(n):
                    for i, camera in enumerate(CAMERAS):
                        rgb[t, i] = decode_rgb(file[f'observation/{camera}/rgb'][t])
                rgb.flush()
                del rgb
                np.savez(temporary / 'joints.npz', states=states[:-1], actions=states[1:])
            record = {'cache_id': cache_id, 'frames': n, 'seen': seen,
                      'rgb_bytes': (temporary / 'rgb.npy').stat().st_size,
                      'joints_sha256': digest(temporary / 'joints.npz')}
            hdf5.unlink()
            write_json(temporary / 'complete.json', record)
            assert not folder.exists()
            temporary.rename(folder)
            episodes[str(episode)] = record
            if episode % 10 == 0 or episode == 49:
                print(f'{task}: 已转换 {episode + 1}/50 条轨迹', flush=True)
    train, valid = split_episodes(task, cfg['validation_episodes'], cfg['seed'])
    return {'archive': filename, 'archive_sha256': archive_hash, 'episodes': episodes,
            'train_episodes': train, 'validation_episodes': valid,
            'train_frames': sum(episodes[str(i)]['frames'] for i in train),
            'validation_frames': sum(episodes[str(i)]['frames'] for i in valid)}


def prepare(cfg):
    import cv2
    import numpy as np
    root = Path(cfg['local_root'])
    root.mkdir(parents=True, exist_ok=True)
    cv2.setNumThreads(1)
    os.environ['HF_HOME'] = str(root / 'hf_cache')
    for key in ['HF_HUB_OFFLINE', 'HF_DATASETS_OFFLINE']:
        os.environ.pop(key, None)
    manifest = {'format': FORMAT, 'data_revision': cfg['data_revision'], 'tasks': {}}
    for task in cfg['tasks']:
        manifest['tasks'][task] = prepare_task(cfg, root, task)
        write_json(root / 'data_manifest.json', manifest)
    states, actions = [], []
    for task, record in manifest['tasks'].items():
        for episode in record['train_episodes']:
            with np.load(root / 'prepared' / task / f'episode{episode}' / 'joints.npz', allow_pickle=False) as z:
                states.append(z['states'])
                actions.append(z['actions'])
    sl, sh = np.quantile(np.concatenate(states), [.01, .99], axis=0).astype(np.float32)
    al, ah = np.quantile(np.concatenate(actions), [.01, .99], axis=0).astype(np.float32)
    stats = {'s_low': sl.tolist(), 's_high': sh.tolist(), 'a_low': al.tolist(), 'a_high': ah.tolist(),
             'method': 'shared_train_only_q01_q99_without_clipping',
             'raw_to_model_slots': [0, 1, 2, 3, 4, 5, 28, 6, 7, 8, 9, 10, 11, 29]}
    write_json(root / 'normalization.json', stats)
    manifest['models'] = download_models(cfg, root)
    manifest['source'] = download_source(cfg, root)
    # 预先核对最长指令，不在正式训练中静默丢掉句尾。
    from transformers import AutoProcessor
    tokenizer = AutoProcessor.from_pretrained(root / 'models/processor', local_files_only=True).tokenizer
    max_tokens = 0
    for record in manifest['tasks'].values():
        for ep in record['episodes'].values():
            for text in ep['seen']:
                prompt = tokenizer.apply_chat_template([{'role': 'user', 'content': text}],
                                                      tokenize=False, add_generation_prompt=False)
                max_tokens = max(max_tokens, len(tokenizer.encode(prompt, add_special_tokens=False)))
    if max_tokens > cfg['max_text_length']:
        raise ValueError(f"最长指令 {max_tokens} tokens，请把 max_text_length 调到至少这个值后重跑本格")
    manifest['max_instruction_tokens'] = max_tokens
    manifest['train_frames'] = sum(t['train_frames'] for t in manifest['tasks'].values())
    write_json(root / 'data_manifest.json', manifest)
    # Drive 只保存轻量清单，不复制几十 GB 的 RGB 缓存；新会话按同一版本重新生成。
    persistent = Path(cfg['save_root']) / cfg['run_name']
    persistent.mkdir(parents=True, exist_ok=True)
    for filename in ['data_manifest.json', 'normalization.json']:
        target = persistent / filename
        if target.exists() and read_json(target) != read_json(root / filename):
            raise ValueError(f'{target} 已有不同实验，请换 run_name；不会覆盖')
        shutil.copy2(root / filename, target)
    print(f"\n准备完成：{len(cfg['tasks'])} 个任务，{manifest['train_frames']} 个训练帧。", flush=True)
    print(f"固定 {cfg['steps']} 次更新 × batch {cfg['effective_batch']} = "
          f"{cfg['steps'] * cfg['effective_batch']} 次窗口抽样。", flush=True)
    print('缓存格式：按 episode 的 RGB224 NPY + 14维关节 NPZ，训练时在线增广。', flush=True)
    return manifest


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    prepare(read_json(parser.parse_args().config))
''', encoding='utf-8')

# ---------- colab10_train.py ----------
(SCRIPTS / 'colab10_train.py').write_text(r'''"""十任务策略主干全量微调。直接进行正式更新，无单独的冒烟训练。"""
import argparse
import gc
import math
import os
import shutil
import time
import uuid
from pathlib import Path

from colab10_common import (BatchCollator, EpisodeStore, UpdateSampler, WindowDataset,
                            identity, read_json, remove_owned, seed_all, selections, write_json)
from colab10_checkpoint import (finalize, load_optimizer, publish, resolve_pointer,
                                save_optimizer, save_weights, verify)
from colab10_model import build_model, optimizer_groups


def to_gpu(batch):
    import torch
    return {k: v.to('cuda', dtype=torch.bfloat16 if k in ['images', 'state', 'actions'] else v.dtype,
                    non_blocking=True) for k, v in batch.items()}


def lr_factor(step, cfg):
    # scheduler.step() 在一次更新后调用；第一个更新使用 1/warmup 的学习率。
    if step < cfg['warmup_steps']:
        return (step + 1) / max(1, cfg['warmup_steps'])
    progress = min(1., (step - cfg['warmup_steps']) / max(1, cfg['steps'] - cfg['warmup_steps']))
    return .1 + .9 * .5 * (1 + math.cos(math.pi * progress))


def signature(cfg, manifest, stats):
    keys = ['tasks', 'data_revision', 'source_revision', 'base_revision', 'processor_revision',
            'seed', 'horizon', 'max_text_length', 'effective_batch', 'steps', 'warmup_steps',
            'lr_vision', 'lr_vlm', 'lr_action', 'weight_decay', 'validation_episodes']
    return identity({'config': {k: cfg[k] for k in keys}, 'data': manifest, 'normalization': stats,
                     'augmentation': 'color_noise_blur_v1', 'training_code': code_identity()})


def code_identity():
    from colab10_common import digest
    return {p.name: digest(p) for p in sorted(Path(__file__).parent.glob('colab10_*.py'))}


def make_loader(cfg, manifest, stats, first, micro):
    import torch
    options = dict(dataset=WindowDataset(cfg['local_root'], manifest, stats, cfg['horizon']),
                   batch_sampler=UpdateSampler(manifest, cfg, first, micro),
                   collate_fn=BatchCollator(Path(cfg['local_root']) / 'models/processor', cfg['max_text_length']),
                   num_workers=cfg['workers'], pin_memory=True)
    if cfg['workers']:
        options.update(multiprocessing_context='spawn', persistent_workers=True, prefetch_factor=2)
    return torch.utils.data.DataLoader(**options)


def backward_update(model, optimizer, iterator, chosen, manifest, cfg, micro):
    import torch
    optimizer.zero_grad(set_to_none=True)
    total_valid = sum(min(cfg['horizon'], manifest['tasks'][t]['episodes'][str(e)]['frames'] - f)
                      for t, e, f, _ in chosen) * 14
    action_log, total_log = torch.zeros((), device='cuda'), torch.zeros((), device='cuda')
    for _ in range(cfg['effective_batch'] // micro):
        cpu = next(iterator)
        valid = float(cpu['joint_mask'].sum())
        batch = to_gpu(cpu)
        # 关闭 autocast 的全模型权重缓存，减少 FP32 训练时的 BF16 常驻副本。
        with torch.autocast('cuda', dtype=torch.bfloat16, cache_enabled=False):
            result = model(**batch)
            action_loss, auxiliary = result[1], result[0] - result[1]
            loss = action_loss * (valid / total_valid) + auxiliary * (micro / cfg['effective_batch'])
        if not torch.isfinite(loss):
            raise FloatingPointError('训练 loss 出现 NaN/Inf，未执行这次参数更新')
        loss.backward()
        action_log += action_loss.detach().float() * (valid / total_valid)
        total_log += loss.detach().float()
    return float(action_log), float(total_log)


def validate(model, cfg, manifest, store, collate):
    import numpy as np
    import torch
    model.eval()
    scores = {}
    try:
        with torch.no_grad(), torch.random.fork_rng(devices=[0]):
            for task_index, task in enumerate(cfg['tasks']):
                rng = np.random.default_rng([cfg['seed'], 999, task_index])
                values = []
                for index in range(cfg['validation_windows_per_task']):
                    episode = int(rng.choice(manifest['tasks'][task]['validation_episodes']))
                    n = manifest['tasks'][task]['episodes'][str(episode)]['frames']
                    key = (task, episode, int(rng.integers(n)), int(rng.integers(2**31)))
                    batch = to_gpu(collate([store.sample(key, augmentation=False)]))
                    # 验证始终 micro=1，固定每个窗口的 flow 时间和噪声，便于跨步比较。
                    torch.manual_seed(cfg['seed'] + task_index * 1000 + index)
                    batch['noise'] = torch.randn_like(batch['actions'])
                    batch['time'] = torch.rand(1, device='cuda') * .998 + .001
                    with torch.autocast('cuda', dtype=torch.bfloat16, cache_enabled=False):
                        result = model(**batch)
                    value = float(result[1])  # 动作损失，不把路由正则混进验证指标。
                    if not math.isfinite(value):
                        raise FloatingPointError('验证 loss 出现 NaN/Inf')
                    values.append(value)
                scores[task] = sum(values) / len(values)
                print(f'  验证 {task}: action_loss={scores[task]:.6f}', flush=True)
        return {'per_task_action_loss': scores, 'mean_action_loss': sum(scores.values()) / len(scores),
                'worst_action_loss': max(scores.values()),
                'note': 'offline flow-matching loss, NOT simulation success rate'}
    finally:
        model.train()


def save_metadata(folder, model, cfg, manifest, stats, metadata):
    from transformers import AutoProcessor
    folder = Path(folder)
    model.config.save_pretrained(folder / 'model_config')
    AutoProcessor.from_pretrained(Path(cfg['local_root']) / 'models/processor',
                                  local_files_only=True).save_pretrained(folder / 'processor')
    for name, value in [('run_config.json', cfg), ('data_manifest.json', manifest),
                        ('normalization.json', stats), ('model_metadata.json', metadata)]:
        write_json(folder / name, value)
    code = folder / 'code'
    code.mkdir(exist_ok=True)
    for p in Path(__file__).parent.glob('colab10_*.py'):
        shutil.copy2(p, code / p.name)


def save_checkpoint(model, optimizer, scheduler, cfg, manifest, stats, metadata, step,
                    micro, best_metric, validation, is_best, fingerprint):
    import torch
    run = Path(cfg['save_root']) / cfg['run_name']
    generation = uuid.uuid4().hex[:8]
    pending = run / f'resume-step-{step:06d}-{generation}.incomplete'
    pending.mkdir(parents=True)
    print(f'保存完整续训状态 step {step}（FP32 权重及 8-bit 优化器，写入 Drive 需要时间）', flush=True)
    save_weights(model, pending / 'weights')
    save_optimizer(optimizer, pending / 'optimizer')
    torch.save({'scheduler': scheduler.state_dict(), 'torch_rng': torch.get_rng_state(),
                'cuda_rng': torch.cuda.get_rng_state()}, pending / 'runtime.pt')
    save_metadata(pending, model, cfg, manifest, stats, metadata)
    write_json(pending / 'training_state.json', {'step': step, 'micro_batch': micro,
        'best_metric': best_metric if math.isfinite(best_metric) else None,
        'signature': fingerprint, 'validation': validation})
    finalize(pending, {'step': step, 'kind': 'full_resume', 'signature': fingerprint})
    if is_best:
        pending_best = run / f'best-step-{step:06d}-{generation}.incomplete'
        pending_best.mkdir()
        # 最佳模型面向 BF16 推理；精确续训必须读取 latest 的 FP32 权重。
        save_weights(model, pending_best / 'weights', dtype=torch.bfloat16)
        save_metadata(pending_best, model, cfg, manifest, stats, metadata)
        write_json(pending_best / 'validation.json', validation)
        finalize(pending_best, {'step': step, 'kind': 'bf16_inference_model', 'signature': fingerprint})
        best = publish(run, pending_best, 'best', step)
        print('最佳推理模型：', best, flush=True)
    latest = publish(run, pending, 'latest', step)
    print('续训状态已完整保存：', latest, flush=True)
    return latest


def train(cfg):
    import fcntl
    import numpy as np
    import torch
    import bitsandbytes as bnb
    root = Path(cfg['local_root'])
    run = Path(cfg['save_root']) / cfg['run_name']
    run.mkdir(parents=True, exist_ok=True)
    # 本地进程锁防止重复点击训练时启动第二个进程。
    lock = (root / 'training.lock').open('w')
    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), '需要支持 BF16 的 NVIDIA GPU'
    memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    assert memory >= 75, f'本配置面向 80GB 显卡，当前只有 {memory:.1f} GiB'
    torch.set_num_threads(4)
    torch.backends.cuda.matmul.allow_tf32 = True
    seed_all(cfg['seed'])
    manifest, stats = read_json(root / 'data_manifest.json'), read_json(root / 'normalization.json')
    fingerprint = signature(cfg, manifest, stats)
    config_path = run / 'experiment.json'
    if config_path.exists():
        assert read_json(config_path)['signature'] == fingerprint, '实验配置/数据/代码变化，请换 run_name，不能静默接旧训练'
    else:
        write_json(config_path, {'signature': fingerprint, 'config': cfg})
    resume = resolve_pointer(run, 'latest') if cfg['resume'] == 'auto' else None
    if cfg['resume'] not in ['auto', None]:
        resume = Path(cfg['resume'])
    if resume is None and resolve_pointer(run, 'latest') is not None:
        raise ValueError('当前实验已有 checkpoint；从头训练请换 run_name')
    first, best_metric, micro = 0, float('inf'), cfg['micro_batch']
    if resume is not None:
        print('核对并读取完整续训状态：', resume, flush=True)
        verify(resume)
        saved = read_json(resume / 'training_state.json')
        assert saved['signature'] == fingerprint
        first = saved['step']
        best_folder = resolve_pointer(run, 'best')
        if best_folder is not None:
            best_metric = read_json(best_folder / 'validation.json')['mean_action_loss']
        micro = min(micro, saved['micro_batch'])
        if first >= cfg['steps']:
            print(f'这个实验已经完成 {first} 步，不会重新训练。新实验请换 run_name。', flush=True)
            return
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    model, metadata = build_model(cfg, root, weights=resume / 'weights' if resume else None,
                                 saved_config=resume / 'model_config' if resume else None)
    groups = optimizer_groups(model, cfg)
    optimizer = bnb.optim.AdamW8bit(groups, betas=(.9, .95), eps=1e-8, min_8bit_size=4096,
                                   block_wise=True, percentile_clipping=100)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: lr_factor(step, cfg))
    if resume:
        load_optimizer(optimizer, resume / 'optimizer')
        state = torch.load(resume / 'runtime.pt', map_location='cpu', weights_only=True)
        scheduler.load_state_dict(state['scheduler'])
        torch.set_rng_state(state['torch_rng'])
        torch.cuda.set_rng_state(state['cuda_rng'])
    print(f"GPU: {torch.cuda.get_device_name(0)} / {memory:.1f} GiB", flush=True)
    print(f"策略主干全量训练：{metadata['trainable_parameters']:,} 参数；无 LoRA，无辅助教师。", flush=True)
    print(f"正式训练 step {first + 1} → {cfg['steps']}，micro={micro}，"
          f"accum={cfg['effective_batch'] // micro}，有效 batch={cfg['effective_batch']}。", flush=True)
    print('FP32 参数/梯度、BF16 计算、8-bit AdamW，梯度检查点已接入联合层和视觉层。', flush=True)
    loader = make_loader(cfg, manifest, stats, first + 1, micro)
    iterator = iter(loader)
    valid_store = EpisodeStore(root, manifest, stats, cfg['horizon'])
    valid_collate = BatchCollator(root / 'models/processor', cfg['max_text_length'])
    local_log = root / 'train.jsonl'
    persistent_log = run / 'train.jsonl'
    # 日志可能已写到 600、完整 checkpoint 只有 500；续训前丢弃未提交分支，避免重复步号。
    for path in [persistent_log, run / 'validation.jsonl']:
        if path.exists():
            temporary = path.with_name(path.name + '.truncate.tmp')
            with path.open(encoding='utf-8') as inp, temporary.open('w', encoding='utf-8') as out:
                for line in inp:
                    try:
                        if __import__('json').loads(line)['step'] <= first:
                            out.write(line.rstrip('\n') + '\n')
                    except (ValueError, KeyError):
                        pass  # 意外中断留下的最后半行不是已提交记录。
            os.replace(temporary, path)
    if persistent_log.exists():
        shutil.copy2(persistent_log, local_log)
    else:
        local_log.write_text('', encoding='utf-8')
    pending_records, completed, last_saved = [], first, first
    unsafe_update = False
    checkpoint_failed = False
    last_validation = None
    update_times = []

    def flush_logs():
        nonlocal pending_records
        if pending_records:
            with persistent_log.open('a', encoding='utf-8') as file:
                file.write(''.join(pending_records))
                file.flush()
            pending_records = []

    def save_current(is_best=False):
        nonlocal last_saved, checkpoint_failed
        optimizer.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()
        flush_logs()
        try:
            save_checkpoint(model, optimizer, scheduler, cfg, manifest, stats, metadata, completed,
                            micro, best_metric, last_validation, is_best, fingerprint)
        except BaseException:
            checkpoint_failed = True
            raise
        last_saved = completed

    try:
        for step in range(first + 1, cfg['steps'] + 1):
            tick = time.monotonic()
            while True:
                retry = False
                try:
                    seed_all(cfg['seed'] + step)
                    chosen = selections(manifest, cfg, step)
                    action_loss, total_loss = backward_update(model, optimizer, iterator, chosen, manifest, cfg, micro)
                except torch.cuda.OutOfMemoryError as exc:
                    exc.__traceback__ = None
                    if micro == 1:
                        raise RuntimeError('micro=1 仍显存不足，当前完整全量配置不能在本运行时容纳；不会自动冻结参数') from None
                    retry = True
                if not retry:
                    break
                optimizer.zero_grad(set_to_none=True)
                del iterator, loader
                gc.collect()
                torch.cuda.empty_cache()
                micro //= 2
                loader = make_loader(cfg, manifest, stats, step, micro)
                iterator = iter(loader)
                print(f'反向显存不足：重放同一次更新，micro={micro}，'
                      f'accum={cfg["effective_batch"] // micro}，有效 batch 保持不变。', flush=True)
            if step == first + 1:
                for role in ['vision', 'vlm', 'action']:
                    has_gradient = any(p.grad is not None and bool(torch.count_nonzero(p.grad))
                        for group in groups if group['role'] == role for p in group['params'])
                    assert has_gradient, f'{role} 没有有效梯度；停止，避免误把冻结训练当成全量训练'
                print('首个正式更新：视觉、VLM、动作分组均有梯度。', flush=True)
            norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],
                                                 1., error_if_nonfinite=True)
            unsafe_update = True
            try:
                optimizer.step()
                torch.cuda.synchronize()
            except torch.cuda.OutOfMemoryError:
                raise RuntimeError('优化器更新阶段显存不足，可能已有部分参数更新；不会保存当前状态。'
                                   '上次完整 latest 保持有效，需要重新规划优化器内存后续训。') from None
            scheduler.step()
            completed = step
            unsafe_update = False
            seconds = time.monotonic() - tick
            update_times.append(seconds)
            record = {'step': step, 'action_loss': action_loss, 'total_loss': total_loss,
                      'grad_norm': float(norm), 'seconds': seconds, 'micro_batch': micro,
                      'effective_batch': cfg['effective_batch'],
                      'peak_gpu_gib': torch.cuda.max_memory_allocated() / 1024**3,
                      'learning_rates': {g['role']: g['lr'] for g in optimizer.param_groups}}
            line = __import__('json').dumps(record) + '\n'
            with local_log.open('a', encoding='utf-8') as file:
                file.write(line)
            pending_records.append(line)
            if step == first + 1 or step % cfg['log_every'] == 0:
                flush_logs()
                average = sum(update_times[-20:]) / len(update_times[-20:])
                hours = average * (cfg['steps'] - step) / 3600
                print(f'step {step}/{cfg["steps"]} | action_loss={action_loss:.5f} | '
                      f'{seconds:.1f}s/update | peak={record["peak_gpu_gib"]:.1f}GiB | '
                      f'剩余纯更新约 {hours:.1f}h（不含验证/保存）', flush=True)
                write_json(run / 'status.json', {'status': 'training', **record})
            is_best = False
            if step % cfg['validate_every'] == 0 or step == cfg['steps']:
                optimizer.zero_grad(set_to_none=True)
                last_validation = {'step': step, **validate(model, cfg, manifest, valid_store, valid_collate)}
                with (run / 'validation.jsonl').open('a', encoding='utf-8') as file:
                    file.write(__import__('json').dumps(last_validation) + '\n')
                is_best = last_validation['mean_action_loss'] < best_metric
                if is_best:
                    best_metric = last_validation['mean_action_loss']
            if step % cfg['save_every'] == 0 or step == cfg['steps'] or is_best:
                save_current(is_best)
        write_json(run / 'status.json', {'status': 'completed', 'step': completed,
                   'best_offline_action_loss': best_metric,
                   'note': '训练完成；仿真成功率需后续在 RoboTwin 测量'})
    except BaseException as exc:
        # 只在已完成一次完整 optimizer.step、且下一步未部分写入参数时保存。
        if completed > last_saved and not unsafe_update and not checkpoint_failed:
            print('训练中断，保存最近完成的更新……', flush=True)
            save_current(False)
        flush_logs()
        write_json(run / 'status.json', {'status': 'interrupted_or_failed', 'step': completed,
                   'saved_step': last_saved, 'message': str(exc), 'partial_update_discarded': unsafe_update})
        raise
    finally:
        flush_logs()
        del iterator, loader
    print('正式训练已完成；最佳 BF16 推理模型与最近完整续训状态均已保存。', flush=True)
    print('结果目录：', run, flush=True)


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    cfg = read_json(parser.parse_args().config)
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    os.environ['OMP_NUM_THREADS'] = '4'
    train(cfg)
''', encoding='utf-8')

# ---------- 安装环境（Notebook 内核和训练子进程相互独立） ----------
ENV = Path('/content/lingbot_full10_env')
PY = ENV / 'bin/python'
if not PY.is_file():
    if sys.version_info[:2] in [(3, 11), (3, 12)]:
        subprocess.run([sys.executable, '-m', 'venv', str(ENV)], check=True)
    else:
        # Colab 日后升级 Python 时，仍给训练程序使用兼容的 Python 3.12。
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.9.5'], check=True)
        subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '--seed', str(ENV)], check=True)

packages = [
    'numpy==1.26.4', 'h5py==3.14.0', 'pillow==11.3.0', 'opencv-python-headless==4.11.0.86',
    'transformers==4.57.3', 'tokenizers==0.22.2', 'huggingface-hub==0.34.3',
    'accelerate==1.7.0', 'safetensors==0.5.3', 'bitsandbytes==0.48.2',
    'torchdata==0.11.0', 'datasets==3.6.0', 'pyarrow==21.0.0', 'pandas==2.3.1',
    'scipy==1.13.1', 'einops==0.8.1', 'pyyaml==6.0.2', 'blobfile==3.2.0',
    'packaging==25.0', 'tqdm==4.67.1', 'psutil==7.0.0', 'pydantic==2.11.7', 'peft==0.15.2',
    'qwen-vl-utils==0.0.11', 'av==15.0.0', 'sentencepiece==0.2.0',
]
environment_id = hashlib.sha256(('torch2.8.0-cu128-tv0.23.0' + repr(packages)).encode()).hexdigest()
marker = ENV / 'installed.txt'
if not marker.is_file() or marker.read_text().strip() != environment_id:
    subprocess.run([str(PY), '-m', 'pip', 'install', '--upgrade', 'pip==25.2'], check=True)
    subprocess.run([str(PY), '-m', 'pip', 'install',
                    'torch==2.8.0', 'torchvision==0.23.0',
                    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
    subprocess.run([str(PY), '-m', 'pip', 'install', *packages], check=True)
    subprocess.run([str(PY), '-m', 'pip', 'check'], check=True)
    marker.write_text(environment_id)
RUN_ENV = os.environ.copy()
RUN_ENV.update(PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='4')
RUN_ENV['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('独立环境和训练程序已准备好。继续运行第 3 格。')


## 3．下载官方材料、转换 500 条示范

自动下载指定 10 个任务各自的 `aloha-agilex_clean_50.zip`，并固定数据版本。
只读取 HDF5 和每条示范的官方 `seen` 指令；不读取 pickle、不读取 `unseen` 指令。

三相机顺序为 head / left / right。对齐仍为 **图像[t]、关节[t] → 关节[t+1] 的绝对目标**，
从每个时刻构造 16 步未来动作窗口，轨迹末尾不足处只填充并屏蔽损失，不跨轨迹。
这次直接转成按轨迹的 NPY/NPZ 缓存供自定义训练器读取，省去 LeRobot Parquet 的额外落盘和再次解码；
不是把旧的单任务数据字段换个任务名称。

每任务固定留出 5 条验证轨迹；共享归一化统计只从 450 条训练轨迹计算。
原始 RGB224 缓存不增广，增广在第 4 格每次取样时在线执行。
仅下载一套 LingBot 基础权重；Qwen 单独只下载处理器/分词器配置。
此格中断后可以重跑，已完整转换的轨迹自动跳过。


In [ ]:
# Colab 不一定显示继承文件描述符的子进程输出，明确逐行转发。
with (WORK / 'prepare_console.log').open('a', encoding='utf-8', buffering=1) as console_log:
    process = subprocess.Popen([str(PY), '-u', str(SCRIPTS / 'colab10_prepare.py'),
                               '--config', str(CONFIG)], env=RUN_ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        console_log.write(line)
    return_code = process.wait()
if return_code:
    raise RuntimeError(f'数据准备退出，返回码 {return_code}；具体错误见上方日志。')


## 4．开始正式全量微调

每 5 次更新，10 个任务各抽到相同数量的窗口；每个任务内部先均匀抽轨迹，再抽起始帧。
50% 样本保持原图，另外 50% 做温和亮度/对比度/饱和度扰动，低概率加入噪声或轻微模糊；
三视角共享颜色参数，不做翻转、换背景或生成式改图。

本格直接训练：**FP32 可训练参数 + BF16 前向计算 + AdamW8bit + 梯度检查点**。
FP32 参数是为了保留小学习率的更新精度；不是把 6B 模型量化为 8-bit，8-bit 指优化器状态。
micro=2、累积=8；反向显存不足时降为 micro=1、累积=16。
首个正式更新会确认视觉/VLM/动作三个分组均有梯度，之后直接继续，不另起试验。

每 500 步和最后一步：逐任务计算固定验证窗口的动作损失，保存完整 latest 续训状态；
任务平均验证动作损失改善时另外保存 best BF16 模型。**验证 loss 不是仿真成功率**，
仿真后续仍在已有 Modal RoboTwin 环境运行。

日志出现 `step 1/10000 | action_loss=...` 表示已完成首次参数更新。
训练完成保存后会出现 `正式训练已完成`。保存大文件时日志暂时不刷新不代表卡住。
点一次停止会尝试保存完整更新边界；强制回收/断电只能恢复至上一次完整保存。


In [ ]:
import fcntl, json, os, signal, subprocess, time
from pathlib import Path
from datetime import datetime, timezone

# 路径和子进程环境不依赖 Notebook 内核中上一次留下的变量。
WORK = Path('/content/lingbot_full10')
SCRIPTS = WORK / 'scripts'
CONFIG = WORK / 'config.json'
PY = Path('/content/lingbot_full10_env/bin/python')
missing = [str(p) for p in [CONFIG, PY, SCRIPTS / 'colab10_train.py', WORK / 'data_manifest.json'] if not p.is_file()]
if missing:
    raise RuntimeError('当前是尚未准备完成的新运行时。需要先恢复 Drive、独立环境和本地数据缓存；'
                       '不会从头启动新实验。缺少：' + ', '.join(missing))
RUN_ENV = os.environ.copy()
RUN_ENV.update(PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='4')
RUN_ENV['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# 只改 Notebook 启动方式，不改训练程序、超参数或 checkpoint。
console_path = WORK / 'training_console.log'
console_path.touch(exist_ok=True)
run_dir = Path(json.loads(CONFIG.read_text())['save_root']) / json.loads(CONFIG.read_text())['run_name']

def training_locked():
    with (WORK / 'training.lock').open('a') as probe:
        try:
            fcntl.flock(probe, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            return True
        fcntl.flock(probe, fcntl.LOCK_UN)
        return False

def brief_progress():
    pointer = run_dir / 'latest.json'
    saved = json.loads(pointer.read_text())['step'] if pointer.exists() else '尚无'
    pending = [(p.name, round(sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1024**3, 2))
               for p in run_dir.glob('*.incomplete')]
    print('最近完整断点：', saved, '；正在写入的目录及 GiB：', pending, flush=True)
    for cache in (WORK / 'resume_cache').glob('*.copying-*'):
        files = [p for p in cache.rglob('*') if p.is_file()]
        print('本地恢复文件准备中：', len(files), '个文件，',
              round(sum(p.stat().st_size for p in files) / 1024**3, 2), 'GiB', flush=True)
    if (WORK / 'runtime_restore_request.json').exists():
        count = len(list((WORK / 'prepared').glob('*/episode*/complete.json')))
        print('本地示范缓存：', count, '/500；此阶段为恢复准备，尚未更新参数。', flush=True)
        recovery_log = WORK / 'runtime_restore_console.log'
        if recovery_log.exists():
            print(''.join(recovery_log.read_text(errors='replace').splitlines(keepends=True)[-3:]), end='', flush=True)

def follow(reader):
    text = reader.read()
    if text:
        print(text, end='', flush=True)

print('训练控制台日志：', console_path, flush=True)
if training_locked():
    print('已有任务持有运行锁：等待训练/保存/本地断点准备结束，不启动第二个训练进程。', flush=True)
    print('若正在保存，中断保存可能丢失尚未发布的进度；请勿删除运行时。', flush=True)
    with console_path.open('r', encoding='utf-8', errors='replace') as reader:
        reader.seek(0, 2)
        last_report = 0
        while training_locked():
            follow(reader)
            if time.monotonic() - last_report >= 60:
                brief_progress()
                last_report = time.monotonic()
            time.sleep(2)
        follow(reader)
    local_log = WORK / 'train.jsonl'
    records = local_log.read_text().splitlines() if local_log.exists() else []
    last_update = json.loads(records[-1])['step'] if records else 0
    pointer = run_dir / 'latest.json'
    saved_step = json.loads(pointer.read_text())['step'] if pointer.exists() else 0
    if saved_step < last_update:
        raise RuntimeError(f'原进程退出但断点只保存到 {saved_step}，内存更新曾到 {last_update}；停止自动启动，请检查保存错误。')
    print('原进程已释放训练锁；现在按原配置读取最新完整断点继续。', flush=True)

if (WORK / 'runtime_restore_request.json').exists() and not (WORK / 'runtime_restore_ready.json').exists():
    raise RuntimeError('新会话恢复准备尚未成功完成；未启动训练。请查看终端中的恢复错误。')

# 训练进程直接写日志文件；Notebook 只负责读取，避免界面断开后管道丢失。
with console_path.open('a', encoding='utf-8', buffering=1) as output:
    output.write('\n=== ' + datetime.now(timezone.utc).isoformat() + ' ===\n')
    with console_path.open('r', encoding='utf-8', errors='replace') as reader:
        reader.seek(0, 2)
        process = subprocess.Popen(
            [str(PY), '-u', str(SCRIPTS / 'colab10_train.py'), '--config', str(CONFIG)],
            env=RUN_ENV, start_new_session=True, stdout=output, stderr=subprocess.STDOUT)
        try:
            while process.poll() is None:
                follow(reader)
                time.sleep(1)
            follow(reader)
        except KeyboardInterrupt:
            print('已请求中断；等待完整更新保存到 Drive。请勿反复点击停止。', flush=True)
            if process.poll() is None:
                process.send_signal(signal.SIGINT)
            last_report = 0
            while process.poll() is None:
                follow(reader)
                if time.monotonic() - last_report >= 60:
                    brief_progress()
                    last_report = time.monotonic()
                time.sleep(1)
            follow(reader)
        return_code = process.wait()
if return_code:
    raise RuntimeError(f'训练进程退出，返回码 {return_code}；具体错误见上方日志。')


## 5．查看训练结果和下次恢复入口

结果目录保留 `latest.json`（完整续训）、`best.json`（最佳推理模型）以及日志。
正常保存完成后自动清理被替代的旧版本，写入期间保留旧的完整版本。
中断造成的 `.incomplete` 文件夹不会作为有效 checkpoint；如果多次写盘失败，
这些残留可能占用空间，需要确认指针后再清理。

`latest` 含 FP32 模型、优化器、学习率调度和 RNG 状态；`best` 是 BF16 推理权重，不含优化器。
两者都带归一化、模型配置、处理器、版本清单和训练代码。
**不要只拿一个权重分片，也不要把完整模型当作 LoRA adapter 加载。**
需要后续评测时使用 best 指向的完整目录；本 Notebook 不会自动启动仿真或报出成功率。


In [ ]:
RUN = Path(SAVE_ROOT) / RUN_NAME
for name in ['status.json', 'latest.json', 'best.json']:
    path = RUN / name
    if path.exists():
        print(name, ':', json.dumps(json.loads(path.read_text()), ensure_ascii=False, indent=2))
best_pointer = RUN / 'best.json'
if best_pointer.exists():
    best = RUN / json.loads(best_pointer.read_text())['directory']
    result = json.loads((best / 'validation.json').read_text())
    print('\n最佳模型的各任务离线动作损失（不是成功率）：')
    for task, value in result['per_task_action_loss'].items():
        print(f'  {task:24s} {value:.6f}')
    print('\n后续评测使用完整目录：', best)
print('\n断线恢复：保持 RUN_NAME 和训练参数，重新连接 A100 后按顺序运行第 1–4 格。')
print('训练结束后如暂时不用 GPU，在 Colab 中断开并删除运行时，释放资源。')


### 参考与边界

- [LingBot-VLA-V2 官方代码](https://github.com/Robbyant/lingbot-vla-v2)
- [本代码固定版本依赖](https://github.com/Robbyant/lingbot-vla-v2/blob/be969b8fd117fb70550c5d4bf4bc328211b5b1b6/requirements.txt)
- [RoboTwin2.0 官方数据仓库](https://huggingface.co/datasets/TianxingChen/RoboTwin2.0)
- [Colab 官方资源和存储说明](https://research.google.com/colaboratory/faq.html)

10,000 步及学习率是本轮正式训练的起点，不是保证达到某个成功率的官方结论。
训练不设置 180 分钟停止条件；Colab 的会话和额度限制仍然有效。
只在这些 10 个任务上训练不代表剩余 40 个任务也已训练。
